### Project Name : **AI Pilot** model

- First let's read our [*CSV*](SmartRocketTrainingData20260507_191122.csv) data file 
We don't use Pandas or Pytorch because we want to understand the under the hood concepts of trining the models this the bigger idea we work for here

In [3]:
import csv

file_path = "SmartRocketTrainingData20260507_191122.csv"
data = []

with open(file_path,'r') as file:
    reader = csv.reader(file)
    header = next(reader)

    for row in reader:
        try:
            presed_row_value = [float(row_value) for row_value in row]
            data.append(presed_row_value)
        except ValueError:
            continue


Now let's filter the Data in inputs X and outputs Y

In [ ]:
X,Y = [],[]
for row in data:
    inputs = [
        row[3], # DistanceX
        row[4], # DistanceY
        row[7], # VelocityX
        row[8], # VelocityY
        row[9], # Angle
        row[10] # AngularVelocity
    ]
    inputs[0] = inputs[0] / 60.0  # div by the max for all
    inputs[1] = inputs[1] / 60.0
    inputs[2] = inputs[2] / 15.0
    inputs[3] = inputs[3] / 15.0
    inputs[4] = inputs[4] / 180
    inputs[5] = inputs[5] / 30

    labels = [
        row[0], # Speed -> Engine thrust
        row[1]  # Steering -> rocket direction
    ]

    labels[0] = labels[0] / 30.0

    X.append(inputs)
    Y.append(labels)


In [19]:
from engine import Value
import random

In [22]:
class Neuron:
    def __init__(self,nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))
    
    def __call__(self,x):
        # wight * x + b
        act = sum((wi*xi for wi , xi in zip(self.w,x)) , self.b)
        out = act.tanh()
        return out
    
    def parameters(self):
        return self.w + [ self.b]

class Layer:
    def __init__(self,nin,nout):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]
       

class MLP:
    def __init__(self,nin,nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i],sz[i+1]) for i in range(len(nouts))]
    
    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]
